# Advanced methods

This section covers advanced methods: inverse function, monomial coefficients, and the product matrix.

## Monomial coefficients

The `monomials` method converts the Chebyshev coefficients of $f$ into the coefficients of the equivalent ordinary polynomial (monomial basis), i.e. it returns $\vec{p}$ such that
```{math}
f(x) = \sum_{n=0}^{N} c_n T_n(\xi) = \sum_{n=0}^{N} p_n \xi^n \;,\qquad \xi = \frac{2(x-a)}{b-a} - 1 \;,
```
It is equivalent to applying the basis's `monomial_matrix` (see {doc}`../api/basis1d`) to the coefficient vector, $\vec{p} = \mat{M}\,\vec{c}$, but computed directly rather than by forming the matrix. Converting to the monomial basis is numerically much less stable than working with Chebyshev coefficients (the monomial basis becomes very ill-conditioned for even moderate order), so this should mainly be used for low-order series or for interoperating with code that expects ordinary polynomial coefficients.

Note that the monomial coefficients $p_n$ are expressed in the mapped variable $\xi \in [-1,1]$ from {eq}`xi_map`, not in $x$ directly.

In [ ]:
import numpy as np
from cheby import RealFunction

f = RealFunction(0.0, 1.0, np.array([1.0, 2.0, 0.3]))
p = f.monomials()
M = f.basis().monomial_matrix()

print('monomials():          ', p)
print('monomial_matrix @ coef:', M @ f.coef)
x = 0.7
xi = 2 * (x - f.start) / (f.end - f.start) - 1  # map x to xi, see xi_map
print('check at x=0.7:', np.polyval(p[::-1], xi), 'vs', f(np.array([x])))

## Product matrix

The `product_matrix` method returns the matrix $\mat{P}_f$ such that, for another function $g$ with coefficients $\vec{g}$, $\mat{P}_f\,\vec{g}$ gives the (possibly truncated) coefficients of the product $f\,g$ from {doc}`arithmetic` -- i.e. multiplication by $f$ becomes an ordinary matrix-vector product. This is useful when building linear operators for spectral discretisations that involve multiplying by a known function (a variable coefficient in a differential equation, for instance).

In [ ]:
g = RealFunction(0.0, 1.0, np.array([0.2, -0.5, 0.1, 0.05]))

P = f.product_matrix(len(g.coef) - 1)
direct = (f * g).coef

print('via product_matrix:', (P @ g.coef)[: len(direct)])
print('via __mul__:       ', direct)

## Functional inverse

For a function that is monotonic over its interval, `inverse` constructs the Chebyshev series of the inverse function $f^{-1}$, defined on $[f(a), f(b)]$ (or $[f(b), f(a)]$ if $f$ is decreasing), by expanding $f^{-1}$ in powers of $f$ using the Chebyshev power series computed with repeated squaring (the same technique as `pow`), and solving a linear least-squares problem for the coefficients that make the composition $f^{-1}(f(x)) = x$ hold. The `N` argument controls the order of the resulting series. This method is currently only available for `RealFunction`.

In [ ]:
monotone = RealFunction(lambda x: x + 0.1 * np.sin(x), 0.0, 2.0)
inv = monotone.inverse(15)

x = np.linspace(0.0, 2.0, 50)
roundtrip = inv(monotone(x))
print('max |inverse(f(x)) - x|:', np.max(np.abs(roundtrip - x)))